# Comparing two REI point clouds

`REIComparison` measures the spatial overlap of two rare-event point clouds
(two metrics/thresholds, or a prediction against a reference), even when they sit
on grids of different spacing (they must share an origin). It reports IoU, Dice,
and containment, matches clusters one-to-one, and writes a classified VTK. Pure
Python (numpy + scipy); runs from `pip install graintrace`.

See :doc:`/algorithms/rei-comparison` and :class:`~graintrace.REIComparison`.

In [ ]:
import numpy as np
import pandas as pd
from graintrace.rei_comparison import REIComparison

## Two clouds of the same blobs at different spacing

Both sample the same three spherical regions. REI 1 uses a fine grid
(`spacing_1`); REI 2 samples the same physical region on a coarser grid
(`spacing_2`). Each blob is a cluster id.

In [ ]:
def blobs(n, spacing, centers, radius, step=1):
    zz, yy, xx = np.mgrid[0:n:step, 0:n:step, 0:n:step]
    rows = []
    for cid, (cx, cy, cz) in enumerate(centers, start=1):
        m = (xx - cx) ** 2 + (yy - cy) ** 2 + (zz - cz) ** 2 <= radius * radius
        rows.append(pd.DataFrame({
            "x": xx[m] * spacing, "y": yy[m] * spacing, "z": zz[m] * spacing,
            "rare_cluster_id": cid,
        }))
    return pd.concat(rows, ignore_index=True)

n = 40
centers = [(10, 10, 10), (25, 25, 20), (15, 30, 30)]
df1 = blobs(n, 1.0, centers, radius=6, step=1)   # fine
df2 = blobs(n, 2.0, centers, radius=6, step=2)   # coarse, same physical blobs
df1.to_csv("rei_A.csv", index=False)
df2.to_csv("rei_B.csv", index=False)
print("REI 1 points:", len(df1), "| REI 2 points:", len(df2))

## Run the comparison

In [ ]:
comp = REIComparison(
    rei_csv_1="rei_A.csv", rei_csv_2="rei_B.csv",
    output_dir="rei_cmp_out",
    spacing_1=1.0, spacing_2=2.0,
    coord_cols=("x", "y", "z"),
    cluster_col="rare_cluster_id",
)
result = comp.run_comparison()
m = result["overlap_metrics"] if "overlap_metrics" in result else result
for k in ("iou", "dice", "containment_1", "containment_2"):
    if k in m:
        print(f"{k:15} {m[k]:.3f}")

The two clouds cover the same blobs, so IoU/Dice are high. `output_dir` also
gets `overlap_metrics.json`, a classified `overlap_cloud.vtk` (only-1 / only-2 /
both), and `cluster_match.csv` (the one-to-one cluster pairing).

**See also**

- Algorithm: :doc:`/algorithms/rei-comparison`
- Producing REI point clouds: :doc:`rare-event-identification`